# ETL Technical Assessment — Análisis Exploratorio de Datos (EDA)

## Objetivo
Explorar y analizar de forma sistemática los archivos fuente de los tres sistemas independientes para descubrir inconsistencias, errores de formato, valores nulos, registros duplicados e infracciones de integridad referencial. Este diagnóstico servirá de base para el diseño del pipeline ETL.

### Características de los Sistemas Fuente
| Sistema | Formato | Codificación | Formato Fechas | Decimales |
| :--- | :--- | :--- | :--- | :--- |
| **SOURCE_A** | CSV | UTF-8 | `YYYY-MM-DD` | Punto decimal (.) |
| **SOURCE_B** | CSV | Latin-1 | `DD/MM/YYYY` | Coma decimal (,) y separador de miles |
| **SOURCE_C** | CSV | UTF-8 | N/A | Símbolo `$` como prefijo |

---
## 0. Configuración del Entorno e Importaciones

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [3]:
# -- Definición de rutas base --
DATA_DIR = Path(r'../../data')
PATH_A = DATA_DIR / 'source_a'
PATH_B = DATA_DIR / 'source_b'
PATH_C = DATA_DIR / 'source_c'

# -- Función de diagnóstico general --
def quick_info(df: pd.DataFrame, name: str) -> None:
    """Imprime métricas básicas de exploración general para un DataFrame."""
    print('=' * 70)
    print(f' Exploración General: {name}')
    print('=' * 70)
    print(f' * Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
    print(f' * Nombres de Columnas: {list(df.columns)}')
    
    # Detección de duplicados totales
    dup_count = df.duplicated().sum()
    print(f' * Filas completamente duplicadas: {dup_count}')
    
    # Análisis general de nulos
    null_counts = df.isnull().sum()
    columns_with_nulls = null_counts[null_counts > 0]
    if not columns_with_nulls.empty:
        print(' * Presencia de valores nulos:')
        for col, count in columns_with_nulls.items():
            pct = round(count / len(df) * 100, 1)
            print(f'   - Columna "{col}": {count} registros nulos ({pct}%)')
    else:
        print(' * No se detectaron valores nulos en el DataFrame.')
    print()

print('Cargado.')

Cargado.


---
## 1. Exploración Sistemática: SOURCE_A

### 1.1 Clientes (`customers.csv`)

In [4]:
a_cust = pd.read_csv(PATH_A / 'customers.csv', encoding='utf-8')
quick_info(a_cust, 'customers.csv')

 Exploración General: customers.csv
 * Dimensiones: 25 filas x 12 columnas
 * Nombres de Columnas: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'address', 'city', 'country', 'registration_date', 'tax_id', 'is_active', 'credit_limit']
 * Filas completamente duplicadas: 0
 * Presencia de valores nulos:
   - Columna "email": 3 registros nulos (12.0%)



In [5]:
print('--- Examinar registros con nulos detectados ---')
# Dado que quick_info reportó nulos en la columna 'email', aislamos las filas
print(a_cust[a_cust['email'].isnull()][['customer_id', 'first_name', 'last_name', 'email']])

print('\n--- Buscar espacios en blanco al inicio o final de las columnas de texto ---')
text_cols = a_cust.select_dtypes(include=['object']).columns
for col in text_cols:
    # Evaluamos si hay cambios al aplicar strip()
    has_whitespaces = a_cust[col].astype(str).str.strip() != a_cust[col].astype(str)
    if has_whitespaces.any():
        print(f' -> Columna "{col}" contiene espacios adicionales en las siguientes filas:')
        print(a_cust[has_whitespaces][['customer_id', col]])

print('\n--- Validar unicidad del identificador principal (customer_id) ---')
dup_ids = a_cust['customer_id'].duplicated().sum()
print(f' Repeticiones de ID de cliente: {dup_ids}')

--- Examinar registros con nulos detectados ---
   customer_id first_name last_name email
7         C008   Jennifer    Taylor   NaN
14        C015     Joseph  Thompson   NaN
21        C022      Susan       Lee   NaN

--- Buscar espacios en blanco al inicio o final de las columnas de texto ---
 -> Columna "first_name" contiene espacios adicionales en las siguientes filas:
   customer_id first_name
11        C012     Nancy 
24        C025     Jason 
 -> Columna "last_name" contiene espacios adicionales en las siguientes filas:
  customer_id last_name
4        C005    Brown 
 -> Columna "email" contiene espacios adicionales en las siguientes filas:
   customer_id email
7         C008   NaN
14        C015   NaN
21        C022   NaN
 -> Columna "address" contiene espacios adicionales en las siguientes filas:
  customer_id         address
4        C005   654 Maple Dr 

--- Validar unicidad del identificador principal (customer_id) ---
 Repeticiones de ID de cliente: 0


### 1.2 Proveedores (`suppliers.csv`)

In [6]:
a_supp = pd.read_csv(PATH_A / 'suppliers.csv', encoding='utf-8')
quick_info(a_supp, 'suppliers.csv')

 Exploración General: suppliers.csv
 * Dimensiones: 20 filas x 11 columnas
 * Nombres de Columnas: ['supplier_id', 'company_name', 'contact_name', 'email', 'phone', 'address', 'city', 'country', 'tax_id', 'payment_terms', 'is_active']
 * Filas completamente duplicadas: 0
 * Presencia de valores nulos:
   - Columna "contact_name": 1 registros nulos (5.0%)
   - Columna "email": 1 registros nulos (5.0%)



In [7]:
print('--- Examinar registros con nulos detectados ---')
# Filtramos de forma general cualquier fila con valores nulos basándonos en el reporte previo
print(a_supp[a_supp.isnull().any(axis=1)][['supplier_id', 'company_name', 'contact_name', 'email']])

print('\n--- Buscar espacios en blanco al inicio o final de las columnas de texto ---')
text_cols = a_supp.select_dtypes(include=['object']).columns
for col in text_cols:
    has_whitespaces = a_supp[col].astype(str).str.strip() != a_supp[col].astype(str)
    if has_whitespaces.any():
        print(f' -> Columna "{col}" contiene espacios adicionales en las siguientes filas:')
        print(a_supp[has_whitespaces][['supplier_id', col]])

--- Examinar registros con nulos detectados ---
   supplier_id        company_name contact_name email
15        S016  Crystal Glassworks          NaN   NaN

--- Buscar espacios en blanco al inicio o final de las columnas de texto ---
 -> Columna "company_name" contiene espacios adicionales en las siguientes filas:
   supplier_id         company_name
12        S013    Falcon Transport 
18        S019       Horizon Fuels 
 -> Columna "contact_name" contiene espacios adicionales en las siguientes filas:
   supplier_id contact_name
15        S016          NaN
 -> Columna "email" contiene espacios adicionales en las siguientes filas:
   supplier_id email
15        S016   NaN


### 1.3 Productos (`products.csv`)

In [8]:
a_prod = pd.read_csv(PATH_A / 'products.csv', encoding='utf-8')
quick_info(a_prod, 'products.csv (SOURCE_A)')

 Exploración General: products.csv (SOURCE_A)
 * Dimensiones: 25 filas x 9 columnas
 * Nombres de Columnas: ['product_id', 'product_name', 'category', 'unit_price', 'supplier_id', 'sku', 'description', 'is_active', 'created_date']
 * Filas completamente duplicadas: 0
 * No se detectaron valores nulos en el DataFrame.



In [9]:
print('--- Comprobación de Integridad Referencial (supplier_id) ---')
# Validamos si existen IDs de proveedores asociados a productos que no se encuentran registrados en suppliers
valid_suppliers = set(a_supp['supplier_id'])
invalid_relations = a_prod[~a_prod['supplier_id'].isin(valid_suppliers)]
if not invalid_relations.empty:
    print(f' ALERTA: Se detectaron {len(invalid_relations)} productos referenciando a proveedores no válidos:')
    print(invalid_relations[['product_id', 'product_name', 'supplier_id']])
else:
    print(' Integridad referencial correcta para proveedores de productos.')

--- Comprobación de Integridad Referencial (supplier_id) ---
 Integridad referencial correcta para proveedores de productos.


### 1.4 Facturas (`invoices.csv`)

In [10]:
a_inv = pd.read_csv(PATH_A / 'invoices.csv', encoding='utf-8')
quick_info(a_inv, 'invoices.csv')

 Exploración General: invoices.csv
 * Dimensiones: 25 filas x 8 columnas
 * Nombres de Columnas: ['invoice_id', 'customer_id', 'invoice_date', 'due_date', 'total_amount', 'tax_amount', 'status', 'payment_method']
 * Filas completamente duplicadas: 0
 * No se detectaron valores nulos en el DataFrame.



In [11]:
print('--- Comprobación de Integridad Referencial (customer_id) ---')
# Verificamos si existen facturas emitidas a clientes que no están en el maestro de clientes
valid_customers = set(a_cust['customer_id'])
orphan_invoices = a_inv[~a_inv['customer_id'].isin(valid_customers)]
if not orphan_invoices.empty:
    print(f' ALERTA: Se encontraron {len(orphan_invoices)} facturas con clientes inexistentes (FK huérfana):')
    print(orphan_invoices[['invoice_id', 'customer_id', 'total_amount', 'status']])
else:
    print(' Integridad referencial correcta: todos los clientes existen.')

--- Comprobación de Integridad Referencial (customer_id) ---
 ALERTA: Se encontraron 1 facturas con clientes inexistentes (FK huérfana):
   invoice_id customer_id  total_amount   status
22       I023        C099         660.0  PENDING


### 1.5 Líneas de Facturas (`invoice_lines.csv`)

In [12]:
a_lines = pd.read_csv(PATH_A / 'invoice_lines.csv', encoding='utf-8')
quick_info(a_lines, 'invoice_lines.csv')

 Exploración General: invoice_lines.csv
 * Dimensiones: 71 filas x 7 columnas
 * Nombres de Columnas: ['line_id', 'invoice_id', 'product_id', 'quantity', 'unit_price', 'discount_pct', 'line_total']
 * Filas completamente duplicadas: 0
 * No se detectaron valores nulos en el DataFrame.



In [13]:
print('--- Validar relación jerárquica con cabeceras de facturas (invoice_id) ---')
# Buscamos líneas de factura cuyo ID de factura padre no exista
valid_invoices = set(a_inv['invoice_id'])
orphan_lines = a_lines[~a_lines['invoice_id'].isin(valid_invoices)]
if not orphan_lines.empty:
    print(f' ALERTA: Se encontraron {len(orphan_lines)} líneas huérfanas de facturas inexistentes:')
    print(orphan_lines)
else:
    print(' Todas las líneas están correctamente asociadas a facturas válidas.')

print('\n--- Validar asociación con productos (product_id) ---')
valid_products = set(a_prod['product_id'])
orphan_product_lines = a_lines[~a_lines['product_id'].isin(valid_products)]
if not orphan_product_lines.empty:
    print(f' ALERTA: Se encontraron {len(orphan_product_lines)} líneas que hacen referencia a productos inexistentes:')
    print(orphan_product_lines)
else:
    print(' Todos los productos referenciados en las líneas de factura son válidos.')

--- Validar relación jerárquica con cabeceras de facturas (invoice_id) ---
 Todas las líneas están correctamente asociadas a facturas válidas.

--- Validar asociación con productos (product_id) ---
 Todos los productos referenciados en las líneas de factura son válidos.


### 1.6 Pagos (`payments.csv`)

In [14]:
a_pay = pd.read_csv(PATH_A / 'payments.csv', encoding='utf-8')
quick_info(a_pay, 'payments.csv')

 Exploración General: payments.csv
 * Dimensiones: 30 filas x 7 columnas
 * Nombres de Columnas: ['payment_id', 'invoice_id', 'payment_date', 'amount', 'payment_method', 'reference_number', 'status']
 * Filas completamente duplicadas: 0
 * No se detectaron valores nulos en el DataFrame.



In [15]:
print('--- Validar referencia de pagos hacia facturas (invoice_id) ---')
orphan_payments = a_pay[~a_pay['invoice_id'].isin(valid_invoices)]
if not orphan_payments.empty:
    print(f' ALERTA: Se detectaron {len(orphan_payments)} pagos para facturas inexistentes:')
    print(orphan_payments)
else:
    print(' Todos los pagos hacen referencia a facturas válidas.')

print('\n--- Identificar facturas con múltiples pagos ---')
payment_counts = a_pay.groupby('invoice_id').size().reset_index(name='cantidad_pagos')
multiple_payment_invoices = payment_counts[payment_counts['cantidad_pagos'] > 1]
if not multiple_payment_invoices.empty:
    print(f' Se detectaron {len(multiple_payment_invoices)} facturas con más de un pago registrado:')
    print(multiple_payment_invoices)
else:
    print(' No se detectaron facturas con múltiples pagos.')

print('\n--- Validar regla de tolerancia cero (Suma de Pagos vs Total Factura) ---')
pay_totals = a_pay.groupby('invoice_id')['amount'].sum().reset_index(name='total_pagado')
inv_amounts = a_inv[['invoice_id', 'total_amount']]
payments_validation = pay_totals.merge(inv_amounts, on='invoice_id', how='left')
payments_validation['exceso'] = (payments_validation['total_pagado'] - payments_validation['total_amount']).round(2)

overpaid_invoices = payments_validation[payments_validation['exceso'] > 0.01]
if not overpaid_invoices.empty:
    print(f' ALERTA: Se detectaron {len(overpaid_invoices)} facturas con pagos que exceden el monto facturado:')
    print(overpaid_invoices)
    print('\nDetalle de pagos asociados a las facturas sobrepagadas:')
    print(a_pay[a_pay['invoice_id'].isin(overpaid_invoices['invoice_id'])].sort_values(['invoice_id', 'payment_date']))
else:
    print(' No se detectaron pagos que excedan el monto total de la factura asociada.')

print('\n--- Buscar pagos potencialmente duplicados por campos de negocio ---')
duplicate_payment_keys = ['invoice_id', 'payment_date', 'amount', 'payment_method', 'reference_number']
duplicate_payments = a_pay[a_pay.duplicated(subset=duplicate_payment_keys, keep=False)]
if not duplicate_payments.empty:
    print(f' ALERTA: Se detectaron {len(duplicate_payments)} pagos potencialmente duplicados:')
    print(duplicate_payments.sort_values(duplicate_payment_keys))
else:
    print(' No se detectaron pagos duplicados con la misma factura, fecha, monto, método y referencia.')

--- Validar referencia de pagos hacia facturas (invoice_id) ---
 Todos los pagos hacen referencia a facturas válidas.

--- Identificar facturas con múltiples pagos ---


 Se detectaron 3 facturas con más de un pago registrado:
   invoice_id  cantidad_pagos
3        I004               2
7        I008               4
18       I019               2

--- Validar regla de tolerancia cero (Suma de Pagos vs Total Factura) ---
 ALERTA: Se detectaron 2 facturas con pagos que exceden el monto facturado:
   invoice_id  total_pagado  total_amount  exceso
7        I008        1350.0         720.0   630.0
18       I019         980.0         490.0   490.0

Detalle de pagos asociados a las facturas sobrepagadas:
   payment_id invoice_id payment_date  amount payment_method reference_number  \
15   PAY-A016       I008   2024-02-23   350.0    CREDIT_CARD      REF-CC-1016   
26   PAY-A027       I008   2024-03-03   300.0    CREDIT_CARD      REF-CC-1027   
27   PAY-A028       I008   2024-03-15   200.0    CREDIT_CARD      REF-CC-1028   
28   PAY-A029       I008   2024-03-28   500.0    CREDIT_CARD      REF-CC-1029   
9    PAY-A010       I019   2024-02-24   490.0           CASH

---
## 2. Exploración Sistemática: SOURCE_B (CRM Legado)

### 2.1 Clientes (`clientes.csv`)

In [16]:
b_cust = pd.read_csv(PATH_B / 'clientes.csv', encoding='latin-1')
quick_info(b_cust, 'clientes.csv')

 Exploración General: clientes.csv
 * Dimensiones: 25 filas x 12 columnas
 * Nombres de Columnas: ['IdCliente', 'NombreCompleto', 'CorreoElectronico', 'Telefono', 'Direccion', 'Ciudad', 'Pais', 'FechaRegistro', 'RFC', 'Activo', 'LimiteCredito', 'TipoCliente']
 * Filas completamente duplicadas: 0
 * Presencia de valores nulos:
   - Columna "CorreoElectronico": 3 registros nulos (12.0%)



In [17]:
print('--- Examinar registros con nulos detectados ---')
print(b_cust[b_cust['CorreoElectronico'].isnull()][['IdCliente', 'NombreCompleto', 'CorreoElectronico']])

print('\n--- Buscar anomalías y espacios en NombreCompleto ---')
# Identificamos si hay espacios adicionales al inicio/final o dobles espacios internos en los nombres
has_extra_spaces = (
    (b_cust['NombreCompleto'].str.strip() != b_cust['NombreCompleto']) |
    (b_cust['NombreCompleto'].str.contains(r'  ', na=False))
)
print(f' Total registros con espaciado incorrecto en nombre: {has_extra_spaces.sum()}')
print(b_cust[has_extra_spaces][['IdCliente', 'NombreCompleto']].head(10))

--- Examinar registros con nulos detectados ---
   IdCliente     NombreCompleto CorreoElectronico
9       B010      Lucia Herrera               NaN
19      B020    Patricia  Luna                NaN
21      B022  Marisol  Cardenas               NaN

--- Buscar anomalías y espacios en NombreCompleto ---
 Total registros con espaciado incorrecto en nombre: 17
   IdCliente     NombreCompleto
2       B003   Roberto  NuÃ±ez 
4       B005        Jose  Lopez
6       B007  Francisco  Torres
9       B010      Lucia Herrera
11      B012      Monica  Ponce
13      B014     Adriana  Mejia
14      B015     Felipe  Orozco
15      B016  Veronica  Padilla
16      B017    Victor  Salazar
17      B018      Carmen  Rivas


### 2.2 Proveedores (`proveedores.csv`)

In [18]:
b_supp = pd.read_csv(PATH_B / 'proveedores.csv', encoding='latin-1')
quick_info(b_supp, 'proveedores.csv')

 Exploración General: proveedores.csv
 * Dimensiones: 20 filas x 12 columnas
 * Nombres de Columnas: ['IdProveedor', 'RazonSocial', 'NombreContacto', 'Correo', 'Telefono', 'DireccionCompleta', 'Ciudad', 'Pais', 'NIT', 'TerminosPago', 'Activo', 'Categoria']
 * Filas completamente duplicadas: 0
 * Presencia de valores nulos:
   - Columna "Correo": 2 registros nulos (10.0%)



In [19]:
print('--- Examinar registros con nulos detectados ---')
print(b_supp[b_supp['Correo'].isnull()][['IdProveedor', 'RazonSocial', 'Correo']])

print('\n--- Buscar espacios en blanco al inicio o final de las columnas de texto ---')
text_cols = b_supp.select_dtypes(include=['object']).columns
for col in text_cols:
    has_whitespaces = b_supp[col].astype(str).str.strip() != b_supp[col].astype(str)
    if has_whitespaces.any():
        print(f' -> Columna "{col}" contiene espacios adicionales en las siguientes filas:')
        print(b_supp[has_whitespaces][['IdProveedor', col]])

--- Examinar registros con nulos detectados ---
   IdProveedor                        RazonSocial Correo
6         P007     Vidrios y Cristales del Norte     NaN
12        P013             Construrama del Centro    NaN

--- Buscar espacios en blanco al inicio o final de las columnas de texto ---
 -> Columna "RazonSocial" contiene espacios adicionales en las siguientes filas:
   IdProveedor                        RazonSocial
6         P007     Vidrios y Cristales del Norte 
8         P009           Seguridad Total Equipos 
10        P011       Herramientas Especializadas 
18        P019           Farmaceutica Nacional   
 -> Columna "Correo" contiene espacios adicionales en las siguientes filas:
   IdProveedor Correo
6         P007    NaN
12        P013    NaN
 -> Columna "DireccionCompleta" contiene espacios adicionales en las siguientes filas:
   IdProveedor      DireccionCompleta
2         P003   Blvd. Transporte 300
7         P008    Blvd. Artesanal 800
12        P013        Blvd. O

### 2.3 Productos (`productos.csv`)

In [20]:
b_prod = pd.read_csv(PATH_B / 'productos.csv', encoding='latin-1')
quick_info(b_prod, 'productos.csv')

 Exploración General: productos.csv
 * Dimensiones: 25 filas x 10 columnas
 * Nombres de Columnas: ['IdProducto', 'NombreProducto', 'Categoria', 'PrecioUnitario', 'IdProveedor', 'CodigoBarras', 'Descripcion', 'Activo', 'FechaCreacion', 'UnidadMedida']
 * Filas completamente duplicadas: 0
 * Presencia de valores nulos:
   - Columna "IdProveedor": 1 registros nulos (4.0%)



In [21]:
print('--- Validar formato del campo numérico PrecioUnitario ---')
# Observamos el tipo de dato y formato del precio
print('Ejemplo de datos originales en PrecioUnitario:')
print(b_prod['PrecioUnitario'].head(5))

print('\n--- Comprobación de Integridad Referencial (IdProveedor) ---')
valid_suppliers_b = set(b_supp['IdProveedor'])
# Buscamos productos que referencian a proveedores que no existen o son nulos
invalid_prov = b_prod[b_prod['IdProveedor'].isnull() | ~b_prod['IdProveedor'].isin(valid_suppliers_b)]
if not invalid_prov.empty:
    print(f' ALERTA: Se detectaron {len(invalid_prov)} productos con proveedor no válido u omitido:')
    print(invalid_prov[['IdProducto', 'NombreProducto', 'IdProveedor']])

--- Validar formato del campo numérico PrecioUnitario ---
Ejemplo de datos originales en PrecioUnitario:
0      299,99
1      149,50
2      550,00
3    3.500,00
4    5.200,00
Name: PrecioUnitario, dtype: str

--- Comprobación de Integridad Referencial (IdProveedor) ---
 ALERTA: Se detectaron 1 productos con proveedor no válido u omitido:
   IdProducto              NombreProducto IdProveedor
23       B024  Extintor Polvo Quimico 5kg         NaN


### 2.4 Facturas (`facturas.csv`)

In [22]:
b_inv = pd.read_csv(PATH_B / 'facturas.csv', encoding='latin-1')
quick_info(b_inv, 'facturas.csv')

 Exploración General: facturas.csv
 * Dimensiones: 25 filas x 10 columnas
 * Nombres de Columnas: ['IdFactura', 'IdCliente', 'FechaFactura', 'FechaVencimiento', 'Subtotal', 'IVA', 'Total', 'Estado', 'MetodoPago', 'Sucursal']
 * Filas completamente duplicadas: 0
 * No se detectaron valores nulos en el DataFrame.



In [23]:
print('--- Comprobación de Integridad Referencial (IdCliente) ---')
valid_clients_b = set(b_cust['IdCliente'])
orphan_invoices_b = b_inv[~b_inv['IdCliente'].isin(valid_clients_b)]
if not orphan_invoices_b.empty:
    print(f' ALERTA: Se detectaron {len(orphan_invoices_b)} facturas emitidas a clientes que no existen:')
    print(orphan_invoices_b[['IdFactura', 'IdCliente', 'Total', 'Estado']])
else:
    print(' Integridad referencial correcta para clientes en facturas.')

print('\n--- Validar concordancia de montos (Subtotal + IVA = Total) ---')
# Comprobamos si la suma de Subtotal + IVA difiere del Total registrado en la factura
computed_total = b_inv['Subtotal'] + b_inv['IVA']
mismatches = b_inv[abs(computed_total - b_inv['Total']) > 0.01]
if not mismatches.empty:
    print(f' Se detectaron {len(mismatches)} facturas con inconsistencias aritméticas en los montos:')
    print(mismatches[['IdFactura', 'Subtotal', 'IVA', 'Total']])
else:
    print(' No se detectaron inconsistencias: Subtotal + IVA coincide con Total.')

--- Comprobación de Integridad Referencial (IdCliente) ---
 ALERTA: Se detectaron 1 facturas emitidas a clientes que no existen:
   IdFactura IdCliente  Total  Estado
23      B024      B099  452.4  PAGADA

--- Validar concordancia de montos (Subtotal + IVA = Total) ---
 No se detectaron inconsistencias: Subtotal + IVA coincide con Total.


### 2.5 Líneas de Facturas (`factura_lineas.csv`)

In [24]:
b_lines = pd.read_csv(PATH_B / 'factura_lineas.csv', encoding='latin-1')
quick_info(b_lines, 'factura_lineas.csv')

 Exploración General: factura_lineas.csv
 * Dimensiones: 71 filas x 7 columnas
 * Nombres de Columnas: ['IdLinea', 'IdFactura', 'IdProducto', 'Cantidad', 'PrecioUnitario', 'Descuento', 'TotalLinea']
 * Filas completamente duplicadas: 0
 * No se detectaron valores nulos en el DataFrame.



In [25]:
print('--- Validar relación jerárquica con cabeceras de facturas (IdFactura) ---')
valid_invoices_b = set(b_inv['IdFactura'])
orphan_lines_b = b_lines[~b_lines['IdFactura'].isin(valid_invoices_b)]
if not orphan_lines_b.empty:
    print(f' ALERTA: Se detectaron {len(orphan_lines_b)} líneas de factura huérfanas:')
    print(orphan_lines_b)
else:
    print(' Relación correcta de líneas con sus cabeceras.')

print('\n--- Validar relación con productos (IdProducto) ---')
valid_products_b = set(b_prod['IdProducto'])
orphan_prod_lines_b = b_lines[~b_lines['IdProducto'].isin(valid_products_b)]
if not orphan_prod_lines_b.empty:
    print(f' ALERTA: Se detectaron {len(orphan_prod_lines_b)} líneas referenciando a productos inexistentes:')
    print(orphan_prod_lines_b)
else:
    print(' Todos los productos en las líneas de factura son válidos.')

--- Validar relación jerárquica con cabeceras de facturas (IdFactura) ---
 Relación correcta de líneas con sus cabeceras.

--- Validar relación con productos (IdProducto) ---
 Todos los productos en las líneas de factura son válidos.


### 2.6 Pagos (`pagos.csv`)

In [26]:
b_pay = pd.read_csv(PATH_B / 'pagos.csv', encoding='latin-1')
quick_info(b_pay, 'pagos.csv')

 Exploración General: pagos.csv
 * Dimensiones: 30 filas x 9 columnas
 * Nombres de Columnas: ['IdPago', 'IdFactura', 'FechaPago', 'Monto', 'MetodoPago', 'Referencia', 'Estado', 'Banco', 'Comision']
 * Filas completamente duplicadas: 0
 * Presencia de valores nulos:
   - Columna "Banco": 6 registros nulos (20.0%)



In [27]:
print('--- Analizar nulos en la columna "Banco" ---')
# Observamos si la ausencia de banco tiene relación lógica con algún método de pago (por ejemplo, efectivo).
null_banks = b_pay[b_pay['Banco'].isnull()]
if not null_banks.empty:
    print(f' Se detectaron {len(null_banks)} pagos sin banco informado.')
    print('Frecuencia de métodos de pago en registros sin banco:')
    print(null_banks['MetodoPago'].value_counts())
else:
    print(' No se detectaron pagos con banco nulo.')

print('\n--- Validar referencia de pagos hacia facturas (IdFactura) ---')
orphan_payments_b = b_pay[~b_pay['IdFactura'].isin(valid_invoices_b)]
if not orphan_payments_b.empty:
    print(f' ALERTA: Se encontraron {len(orphan_payments_b)} pagos asociados a facturas inexistentes:')
    print(orphan_payments_b)
else:
    print(' Todos los pagos hacen referencia a facturas válidas.')

print('\n--- Identificar facturas con múltiples pagos ---')
payment_counts_b = b_pay.groupby('IdFactura').size().reset_index(name='cantidad_pagos')
multiple_payment_invoices_b = payment_counts_b[payment_counts_b['cantidad_pagos'] > 1]
if not multiple_payment_invoices_b.empty:
    print(f' Se detectaron {len(multiple_payment_invoices_b)} facturas con más de un pago registrado:')
    print(multiple_payment_invoices_b)
else:
    print(' No se detectaron facturas con múltiples pagos.')

print('\n--- Validar regla de tolerancia cero (Suma acumulada vs Total Factura) ---')
# Convertimos primero el campo 'Monto' del formato mexicano (Ej: "1.450,00" -> 1450.00).
def parse_mx_decimal(s):
    return pd.to_numeric(s.astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False), errors='coerce')

b_pay['monto_cleaned'] = parse_mx_decimal(b_pay['Monto'])
pay_sums_b = b_pay.groupby('IdFactura')['monto_cleaned'].sum().reset_index(name='total_pagado')
inv_totals_b = b_inv[['IdFactura', 'Total']]
merged_validation = pay_sums_b.merge(inv_totals_b, on='IdFactura', how='left')
merged_validation['exceso'] = (merged_validation['total_pagado'] - merged_validation['Total']).round(2)

overpaid_invoices_b = merged_validation[merged_validation['exceso'] > 0.01]
if not overpaid_invoices_b.empty:
    print(f' ALERTA: Se detectaron {len(overpaid_invoices_b)} facturas con pagos que exceden el monto facturado:')
    print(overpaid_invoices_b)
    print('\nDetalle de pagos asociados a las facturas sobrepagadas:')
    print(b_pay[b_pay['IdFactura'].isin(overpaid_invoices_b['IdFactura'])].sort_values(['IdFactura', 'FechaPago']))
else:
    print(' No se detectaron pagos que excedan el monto total de la factura asociada.')

print('\n--- Buscar pagos potencialmente duplicados por campos de negocio ---')
duplicate_payment_keys_b = ['IdFactura', 'FechaPago', 'Monto', 'MetodoPago', 'Referencia']
duplicate_payments_b = b_pay[b_pay.duplicated(subset=duplicate_payment_keys_b, keep=False)]
if not duplicate_payments_b.empty:
    print(f' ALERTA: Se detectaron {len(duplicate_payments_b)} pagos potencialmente duplicados:')
    print(duplicate_payments_b.sort_values(duplicate_payment_keys_b))
else:
    print(' No se detectaron pagos duplicados con la misma factura, fecha, monto, método y referencia.')

--- Analizar nulos en la columna "Banco" ---
 Se detectaron 6 pagos sin banco informado.
Frecuencia de métodos de pago en registros sin banco:
MetodoPago
EFECTIVO    6
Name: count, dtype: int64

--- Validar referencia de pagos hacia facturas (IdFactura) ---
 Todos los pagos hacen referencia a facturas válidas.

--- Identificar facturas con múltiples pagos ---
 Se detectaron 2 facturas con más de un pago registrado:
  IdFactura  cantidad_pagos
3      B004               2
7      B008               5

--- Validar regla de tolerancia cero (Suma acumulada vs Total Factura) ---
 ALERTA: Se detectaron 1 facturas con pagos que exceden el monto facturado:
  IdFactura  total_pagado  Total  exceso
7      B008        2050.0  754.0  1296.0

Detalle de pagos asociados a las facturas sobrepagadas:
      IdPago IdFactura   FechaPago   Monto       MetodoPago      Referencia  \
26  PAG-B027      B008  03/03/2024  350,00  TARJETA_CREDITO  REF-CC-MX-2027   
29  PAG-B030      B008  03/04/2024  500,00  TARJ

---
## 3. Exploración Sistemática: SOURCE_C (Catálogo del Portal)

### 3.1 Catálogo de Productos (`products_catalog.csv`)

In [28]:
c_cat = pd.read_csv(PATH_C / 'products_catalog.csv', encoding='utf-8')
quick_info(c_cat, 'products_catalog.csv')

 Exploración General: products_catalog.csv
 * Dimensiones: 20 filas x 8 columnas
 * Nombres de Columnas: ['product_code', 'name', 'category', 'price', 'supplier_code', 'description', 'active', 'uom']
 * Filas completamente duplicadas: 0
 * Presencia de valores nulos:
   - Columna "supplier_code": 1 registros nulos (5.0%)



In [29]:
print('--- Examinar registros con nulos detectados ---')
print(c_cat[c_cat.isnull().any(axis=1)][['product_code', 'name', 'supplier_code']])

print('\n--- Analizar formato de la columna de precio (price) ---')
print('Ejemplo de valores originales de precio:')
print(c_cat['price'].head(5))

print('\n--- Buscar espacios en blanco al inicio o final de las columnas de texto ---')
text_cols = c_cat.select_dtypes(include=['object']).columns
for col in text_cols:
    has_whitespaces = c_cat[col].astype(str).str.strip() != c_cat[col].astype(str)
    if has_whitespaces.any():
        print(f' -> Columna "{col}" contiene espacios adicionales en las siguientes filas:')
        print(c_cat[has_whitespaces][['product_code', col]])

--- Examinar registros con nulos detectados ---
   product_code                        name supplier_code
16     CAT-C017     Grapadora Industrial              NaN

--- Analizar formato de la columna de precio (price) ---
Ejemplo de valores originales de precio:
0     $89.99
1     $34.50
2     $22.00
3    $180.00
4     $15.99
Name: price, dtype: str

--- Buscar espacios en blanco al inicio o final de las columnas de texto ---
 -> Columna "name" contiene espacios adicionales en las siguientes filas:
   product_code                        name
4      CAT-C005              Cable HDMI 5m 
16     CAT-C017     Grapadora Industrial   
 -> Columna "supplier_code" contiene espacios adicionales en las siguientes filas:
   product_code supplier_code
16     CAT-C017           NaN


---
## 4. Análisis Cruzado de Entidades entre Fuentes

### 4.1 Coincidencia de Clientes (Detección de Duplicados por Identificador Fiscal: tax_id / RFC)

In [30]:
a_tax_ids = set(a_cust['tax_id'].dropna().str.strip())
b_rfc_ids = set(b_cust['RFC'].dropna().str.strip())

cross_registered_customers = a_tax_ids.intersection(b_rfc_ids)
print(f'Clientes que comparten el mismo ID fiscal en ambos sistemas (A y B): {len(cross_registered_customers)}')
if cross_registered_customers:
    print('Detalle de coincidencia:')
    matched_a = a_cust[a_cust['tax_id'].str.strip().isin(cross_registered_customers)][['customer_id', 'first_name', 'last_name', 'tax_id']]
    matched_b = b_cust[b_cust['RFC'].str.strip().isin(cross_registered_customers)][['IdCliente', 'NombreCompleto', 'RFC']]
    print('\nRegistros en SOURCE_A:')
    print(matched_a)
    print('\nRegistros en SOURCE_B:')
    print(matched_b)

Clientes que comparten el mismo ID fiscal en ambos sistemas (A y B): 2
Detalle de coincidencia:

Registros en SOURCE_A:
  customer_id first_name last_name  tax_id
2        C003     Robert  Williams  TX-003
6        C007     Thomas     Moore  TX-007

Registros en SOURCE_B:
  IdCliente     NombreCompleto     RFC
2      B003   Roberto  NuÃ±ez   TX-003
6      B007  Francisco  Torres  TX-007


### 4.2 Coincidencia de Proveedores (Detección de Duplicados por Identificador Fiscal: tax_id / NIT)

In [31]:
a_supp_tax = set(a_supp['tax_id'].dropna().str.strip())
b_supp_nit = set(b_supp['NIT'].dropna().str.strip())

cross_registered_suppliers = a_supp_tax.intersection(b_supp_nit)
print(f'Proveedores que comparten el mismo ID fiscal en ambos sistemas (A y B): {len(cross_registered_suppliers)}')
if cross_registered_suppliers:
    print('Detalle de coincidencia:')
    print('IDs fiscales comunes:', cross_registered_suppliers)

Proveedores que comparten el mismo ID fiscal en ambos sistemas (A y B): 0


### 4.3 Coincidencia de Productos por Código de Identificación (SKU / Código de Barras / Código de Catálogo)

In [32]:
sku_a = set(a_prod['sku'].dropna().str.strip())
sku_b = set(b_prod['CodigoBarras'].dropna().str.strip())
sku_c = set(c_cat['product_code'].dropna().str.strip())

print(f' Códigos únicos en SOURCE_A (sku): {len(sku_a)}')
print(f' Códigos únicos en SOURCE_B (CodigoBarras): {len(sku_b)}')
print(f' Códigos únicos en SOURCE_C (product_code): {len(sku_c)}')
print(f' Intersección A <-> B por código directo: {len(sku_a.intersection(sku_b))}')
print(f' Intersección A <-> C por código directo: {len(sku_a.intersection(sku_c))}')
print(f' Intersección B <-> C por código directo: {len(sku_b.intersection(sku_c))}')
print('\n -> Conclusión: Al no existir intersección directa de códigos entre sistemas debido a formatos independientes,')
print('    se cargarán los productos de forma independiente etiquetando su procedencia.')

 Códigos únicos en SOURCE_A (sku): 25
 Códigos únicos en SOURCE_B (CodigoBarras): 25
 Códigos únicos en SOURCE_C (product_code): 20
 Intersección A <-> B por código directo: 0
 Intersección A <-> C por código directo: 0
 Intersección B <-> C por código directo: 0

 -> Conclusión: Al no existir intersección directa de códigos entre sistemas debido a formatos independientes,
    se cargarán los productos de forma independiente etiquetando su procedencia.


---
## 5. Controles Globales Requeridos por Reglas de Calidad

Registros sin ID de origen y fechas nulas o inválidas.

In [33]:
print('--- Paso 1: Validar registros sin ID de origen (tolerancia cero) ---')
source_id_checks = [
    ('SOURCE_A', 'customers', a_cust, 'customer_id'),
    ('SOURCE_A', 'suppliers', a_supp, 'supplier_id'),
    ('SOURCE_A', 'products', a_prod, 'product_id'),
    ('SOURCE_A', 'invoices', a_inv, 'invoice_id'),
    ('SOURCE_A', 'invoice_lines', a_lines, 'line_id'),
    ('SOURCE_A', 'payments', a_pay, 'payment_id'),
    ('SOURCE_B', 'clientes', b_cust, 'IdCliente'),
    ('SOURCE_B', 'proveedores', b_supp, 'IdProveedor'),
    ('SOURCE_B', 'productos', b_prod, 'IdProducto'),
    ('SOURCE_B', 'facturas', b_inv, 'IdFactura'),
    ('SOURCE_B', 'factura_lineas', b_lines, 'IdLinea'),
    ('SOURCE_B', 'pagos', b_pay, 'IdPago'),
    ('SOURCE_C', 'products_catalog', c_cat, 'product_code'),
]

missing_source_id_findings = []
for source, table, df, id_col in source_id_checks:
    missing_mask = df[id_col].isnull() | (df[id_col].astype(str).str.strip() == '')
    if missing_mask.any():
        affected = df[missing_mask]
        missing_source_id_findings.append((source, table, id_col, len(affected)))
        print(f' ALERTA: {source}.{table} contiene {len(affected)} registros sin {id_col}:')
        print(affected)

if not missing_source_id_findings:
    print(' No se detectaron registros sin ID de origen en las tablas fuente revisadas.')

print('\n--- Paso 2: Validar fechas nulas o inválidas según formato esperado ---')
date_checks = [
    ('SOURCE_A', 'customers', a_cust, 'registration_date', '%Y-%m-%d'),
    ('SOURCE_A', 'products', a_prod, 'created_date', '%Y-%m-%d'),
    ('SOURCE_A', 'invoices', a_inv, 'invoice_date', '%Y-%m-%d'),
    ('SOURCE_A', 'invoices', a_inv, 'due_date', '%Y-%m-%d'),
    ('SOURCE_A', 'payments', a_pay, 'payment_date', '%Y-%m-%d'),
    ('SOURCE_B', 'clientes', b_cust, 'FechaRegistro', '%d/%m/%Y'),
    ('SOURCE_B', 'facturas', b_inv, 'FechaFactura', '%d/%m/%Y'),
    ('SOURCE_B', 'facturas', b_inv, 'FechaVencimiento', '%d/%m/%Y'),
    ('SOURCE_B', 'pagos', b_pay, 'FechaPago', '%d/%m/%Y'),
]

invalid_date_findings = []
for source, table, df, date_col, date_format in date_checks:
    parsed_dates = pd.to_datetime(df[date_col], format=date_format, errors='coerce')
    invalid_mask = df[date_col].isnull() | (df[date_col].astype(str).str.strip() == '') | parsed_dates.isnull()
    if invalid_mask.any():
        affected = df[invalid_mask]
        invalid_date_findings.append((source, table, date_col, len(affected)))
        print(f' ALERTA: {source}.{table}.{date_col} tiene {len(affected)} fechas nulas o inválidas:')
        print(affected)

if not invalid_date_findings:
    print(' No se detectaron fechas nulas o inválidas en los campos evaluados.')


--- Paso 1: Validar registros sin ID de origen (tolerancia cero) ---
 No se detectaron registros sin ID de origen en las tablas fuente revisadas.

--- Paso 2: Validar fechas nulas o inválidas según formato esperado ---
 No se detectaron fechas nulas o inválidas en los campos evaluados.


---
## 6. Resumen Consolidado de Calidad de Datos

| Fuente | Tabla | Inconsistencia | Registros Afectados | Tolerancia | Acción Recomendada |
| :--- | :--- | :--- | :---: | :--- | :--- |
| **SOURCE_A** | customers | email nulo | 3 | ALTA | Completar con no_disponible@placeholder.com |
| **SOURCE_A** | customers | Espacios en blanco adicionales | 4 | MEDIA | Aplicar TRIM automático en la carga |
| **SOURCE_A** | suppliers | email / contact_name omitidos | 2 | ALTA | Completar con placeholders estándar según columna afectada |
| **SOURCE_A** | suppliers | Espacios en blanco en compañía | 2 | MEDIA | Aplicar TRIM automático |
| **SOURCE_A** | invoices | ID de cliente C099 no existe en maestro | 1 | CERO | Excluir factura y sus registros dependientes |
| **SOURCE_A** | payments | Facturas con pagos que exceden el monto total (I008, I019) | *len(overpaid_invoices)* | CERO | Excluir pagos conflictivos y reportar motivo |
| **SOURCE_A** | payments | Facturas con múltiples pagos registrados | *len(multiple_payment_invoices)* | CRITERIO | Aceptar solo si la suma acumulada no excede el total facturado |
| **SOURCE_A** | payments | Pagos duplicados por factura, fecha, monto, método y referencia | *len(duplicate_payments)* | CRITERIO | Deduplicar solo si coincide la llave de negocio definida |
| **SOURCE_B** | clientes | Correos electrónicos faltantes | 3 | ALTA | Completar con placeholders |
| **SOURCE_B** | clientes | Espacios adicionales y dobles en nombres | *int(has_extra_spaces.sum())* | MEDIA | Normalizar espaciado (TRIM y reemplazo doble a simple) |
| **SOURCE_B** | proveedores | Correo electrónico faltante | 2 | ALTA | Completar con placeholder |
| **SOURCE_B** | proveedores | Espacios adicionales en RazonSocial | 4 | MEDIA | Aplicar TRIM automático |
| **SOURCE_B** | productos | PrecioUnitario en formato no apto (separadores miles/coma) | 25 | MEDIA | Cast a DECIMAL eliminando puntos e intercambiando comas |
| **SOURCE_B** | productos | Identificador de proveedor omitido o nulo | 1 | ALTA | Permitir supplier_id NULL y reportar proveedor no informado |
| **SOURCE_B** | facturas | ID de cliente B099 no existe en maestro | 1 | CERO | Excluir factura y sus registros dependientes |
| **SOURCE_B** | pagos | Facturas con pagos que exceden el monto total (B008) | *len(overpaid_invoices_b)* | CERO | Excluir pagos conflictivos y reportar motivo |
| **SOURCE_B** | pagos | Facturas con múltiples pagos registrados | *len(multiple_payment_invoices_b)* | CRITERIO | Aceptar solo si la suma acumulada no excede el total facturado |
| **SOURCE_B** | pagos | Monto en formato decimal con comas | 30 | MEDIA | Normalizar conversión a flotante/decimal |
| **SOURCE_B** | pagos | Banco nulo en transacciones de Efectivo | *len(null_banks)* | MEDIA | Esperado de negocio - aceptar NULL y documentar |
| **SOURCE_B** | pagos | Pagos duplicados por factura, fecha, monto, método y referencia | *len(duplicate_payments_b)* | CRITERIO | Deduplicar solo si coincide la llave de negocio definida |
| **SOURCE_C** | products_catalog | Precio con formato no apto (símbolo $) | 20 | MEDIA | Limpiar prefijo de moneda y cast a DECIMAL |
| **SOURCE_C** | products_catalog | Identificador de proveedor omitido o nulo | 1 | ALTA | Permitir supplier_id NULL y reportar proveedor no informado |
| **SOURCE_C** | products_catalog | Espacios adicionales en campo name | 2 | MEDIA | Aplicar TRIM automático |
| **GLOBAL** | all_sources | Registros sin ID de origen | *sum(item[3] for item in missing_source_id_findings)* | CERO | Excluir cualquier registro sin ID de origen |
| **GLOBAL** | date_fields | Fechas nulas o inválidas | *sum(item[3] for item in invalid_date_findings)* | ALTA | Usar 1900-01-01 para fechas nulas y corregir/revisar formatos inválidos |
| **CROSS** | customers | Duplicidad fiscal A <-> B | *len(cross_registered_customers)* | CRITERIO | Cargar ambas entidades manteniendo código origen y sistema diferenciado |
| **CROSS** | suppliers | Duplicidad fiscal A <-> B | *len(cross_registered_suppliers)* | CRITERIO | Cargar ambas entidades manteniendo código origen y sistema diferenciado si aparecen coincidencias |
| **CROSS** | products | Sin SKU común en catálogo unificado | *len(sku_a) + len(sku_b) + len(sku_c)* | CRITERIO | Cargar la totalidad de productos marcando su procedencia |